In [11]:
# Installation des librairies nécessaires
!pip install duckdb pandas matplotlib seaborn plotly -q

import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Setup terminé ! DuckDB est prêt à l'emploi.")
print(f"Version DuckDB : {duckdb.__version__}")
print(f"Version Pandas : {pd.__version__}")

Setup terminé ! DuckDB est prêt à l'emploi.
Version DuckDB : 1.4.4
Version Pandas : 2.3.3


In [12]:
import pandas as pd

In [13]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR

sorted([p.name for p in RAW_DIR.glob("*")])


['.gitkeep',
 'olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [14]:
import pandas as pd

- Analysons la table customers

In [15]:
con = duckdb.connect()

csv_path = RAW_DIR / "olist_customers_dataset.csv"

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path.as_posix()}')
    LIMIT 5
""").df()


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [16]:
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE customers AS
    SELECT *
    FROM read_csv_auto('{csv_path.as_posix()}');
""")

nulls_by_col = con.execute("""
    SELECT
      SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
      SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END) AS customer_unique_id_nulls,
      SUM(CASE WHEN customer_zip_code_prefix IS NULL THEN 1 ELSE 0 END) AS customer_zip_code_prefix_nulls,
      SUM(CASE WHEN customer_city IS NULL THEN 1 ELSE 0 END) AS customer_city_nulls,
      SUM(CASE WHEN customer_state IS NULL THEN 1 ELSE 0 END) AS customer_state_nulls
    FROM customers
""").df()

nulls_by_col



,customer_id_nulls,customer_unique_id_nulls,customer_zip_code_prefix_nulls,customer_city_nulls,customer_state_nulls
0,0.0,0.0,0.0,0.0,0.0


Au regard des resultats, nous nous rendons compte qu'il existe aucune valeur manque pour cette table.

In [17]:
dups = con.execute("""
SELECT
  (SELECT COUNT(*) FROM (SELECT customer_id FROM customers GROUP BY customer_id HAVING COUNT(*) > 1)) AS customer_id_dups,
  (SELECT COUNT(*) FROM (SELECT customer_unique_id FROM customers GROUP BY customer_unique_id HAVING COUNT(*) > 1)) AS customer_unique_id_dups,
  (SELECT COUNT(*) FROM (SELECT customer_zip_code_prefix FROM customers GROUP BY customer_zip_code_prefix HAVING COUNT(*) > 1)) AS customer_zip_code_prefix_dups,
  (SELECT COUNT(*) FROM (SELECT customer_city FROM customers GROUP BY customer_city HAVING COUNT(*) > 1)) AS customer_city_dups,
  (SELECT COUNT(*) FROM (SELECT customer_state FROM customers GROUP BY customer_state HAVING COUNT(*) > 1)) AS customer_state_dups
""").df()

dups


,customer_id_dups,customer_unique_id_dups,customer_zip_code_prefix_dups,customer_city_dups,customer_state_dups
0,0,2997,11982,2975,27


En se focalisant sur ces resultats, nous pourrons dire qu'il existe plusieurs doublons. Or en réalité, plusieurs clients peuvent etre issus de la meme ville, region et peuvent avoir le meme code postal. Pour cela, analysons plutot les lignes et non les variables.

In [18]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state)) AS distinct_rows,
  COUNT(*) - COUNT(DISTINCT (customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state)) AS duplicate_rows
FROM customers
""").df()


,total_rows,distinct_rows,duplicate_rows
0,99441,99441,0


In [19]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

out_path_cus = (PROCESSED_DIR / "customers_clean.csv").as_posix()


con.execute(f"""
COPY ( 
  SELECT *  FROM customers
) TO '{out_path_cus}' (FORMAT CSV, HEADER, DELIMITER ',');
""")

Après l'analyse des doublons sur la base des lignes, nous nous apercevons qu'il n'existe aucun doublon

- Puis Analysons la table geolocation

In [20]:
con = duckdb.connect()

csv_path_geo = RAW_DIR / "olist_geolocation_dataset.csv"

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_geo.as_posix()}')
    LIMIT 5
""").df()


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [21]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE geolocation AS
SELECT *
FROM read_csv_auto(
  '{csv_path_geo.as_posix()}',
  strict_mode=false,
  ignore_errors=true
);
""")


nulls_by_col = con.execute("""
    SELECT
      SUM(CASE WHEN geolocation_zip_code_prefix IS NULL THEN 1 ELSE 0 END) AS geolocation_zip_code_prefix_nulls,
      SUM(CASE WHEN geolocation_lat IS NULL THEN 1 ELSE 0 END) AS geolocation_lat_nulls,
      SUM(CASE WHEN geolocation_lng IS NULL THEN 1 ELSE 0 END) AS geolocation_lng_nulls,
      SUM(CASE WHEN geolocation_city IS NULL THEN 1 ELSE 0 END) AS geolocation_city_nulls,
      SUM(CASE WHEN geolocation_state IS NULL THEN 1 ELSE 0 END) AS geolocation_state_nulls
    FROM geolocation
""").df()

nulls_by_col

,geolocation_zip_code_prefix_nulls,geolocation_lat_nulls,geolocation_lng_nulls,geolocation_city_nulls,geolocation_state_nulls
0,0.0,0.0,0.0,0.0,0.0


Pareil pour cette table, nous n'avons aucune valeur manquante.

In [22]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
  )) AS distinct_rows,
  COUNT(*) - COUNT(DISTINCT (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
  )) AS duplicate_rows
FROM geolocation
""").df()


,total_rows,distinct_rows,duplicate_rows
0,1000163,738332,261831


- A ce niveau, nous avons terminé l'exploration initiale des ensembles de données "geolocation". Nous avons chargé les données dans des tables temporaires DuckDB, vérifié la présence de valeurs manquantes et identifié les doublons.

In [23]:
con.execute("""
CREATE OR REPLACE TEMP TABLE geolocation_dedup AS
SELECT DISTINCT
  geolocation_zip_code_prefix,
  geolocation_lat,
  geolocation_lng,
  geolocation_city,
  geolocation_state
FROM geolocation
""")
con.execute("""
SELECT COUNT(*) AS deduped_rows FROM geolocation_dedup
""").df()   

,deduped_rows
0,738332


In [24]:
out_path_geo = (PROCESSED_DIR / "geolocation_clean.csv").as_posix()

con.execute("""
CREATE OR REPLACE TEMP TABLE geolocation_dedup AS
SELECT DISTINCT
  geolocation_zip_code_prefix,
  geolocation_lat,
  geolocation_lng,
  geolocation_city,
  geolocation_state
FROM geolocation
""")

con.execute(f"""
COPY geolocation_dedup
TO '{out_path_geo}' (FORMAT CSV, HEADER, DELIMITER ',');
""")

Après suppression des doublons, nous avons enregistré la table traité dans notre base data/processed

- Passons à la table olist_order_items_dataset.csv

In [25]:
con = duckdb.connect()

csv_path_order_it = RAW_DIR / "olist_order_items_dataset.csv"

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_order_it.as_posix()}')
    LIMIT 5
""").df()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [26]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE order_items AS
SELECT *
FROM read_csv_auto('{csv_path_order_it.as_posix()}');
""")
con.execute("""
SELECT
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS order_item_id_nulls,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulls,
  SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS seller_id_nulls,
  SUM(CASE WHEN shipping_limit_date IS NULL THEN 1 ELSE 0 END) AS shipping_limit_date_nulls,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS price_nulls,
  SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS freight_value_nulls
FROM order_items
""").df()


,order_id_nulls,order_item_id_nulls,product_id_nulls,seller_id_nulls,shipping_limit_date_nulls,price_nulls,freight_value_nulls
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
con.execute("""
SELECT
  COUNT(*) FILTER (WHERE price <= 0) AS invalid_price,
  COUNT(*) FILTER (WHERE freight_value < 0) AS invalid_freight
FROM order_items
""").df()


,invalid_price,invalid_freight
0,0,0


In [28]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
  )) AS distinct_rows,
  COUNT(*) - COUNT(DISTINCT (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
  )) AS duplicate_rows
FROM order_items
""").df()


,total_rows,distinct_rows,duplicate_rows
0,112650,112650,0


In [29]:
out_path_order_it = (PROCESSED_DIR / "order_items_clean.csv").as_posix()

con.execute(f"""
COPY (
  SELECT DISTINCT *
  FROM order_items
) TO '{out_path_order_it}' (FORMAT CSV, HEADER, DELIMITER ',');
""")

Après analyse appronfondie de cette table, nous nous sommes rendus compte qu'elle en presente aucun problème tant coté valeurs manquantes que sur l'aspect des doublons. De plus, les variables comme prix refletent la réalité car aucun prix n'est inferieur à Zéro.

- Passons à l'analyse de la table order_payments

In [30]:
con = duckdb.connect()

csv_path_order_pay = RAW_DIR / "olist_order_payments_dataset.csv"

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_order_pay.as_posix()}')
    LIMIT 5
""").df()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [31]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE order_payments AS
SELECT *
FROM read_csv_auto('{csv_path_order_pay.as_posix()}');
""")
con.execute("""
SELECT
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN payment_sequential IS NULL THEN 1 ELSE 0 END) AS payment_sequential_nulls,
  SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls,
  SUM(CASE WHEN payment_installments IS NULL THEN 1 ELSE 0 END) AS payment_installments_nulls,
  SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS payment_value_nulls
FROM order_payments
""").df()

,order_id_nulls,payment_sequential_nulls,payment_type_nulls,payment_installments_nulls,payment_value_nulls
0,0.0,0.0,0.0,0.0,0.0


In [32]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
  )) AS distinct_rows,
  COUNT(*) - COUNT(DISTINCT (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
  )) AS duplicate_rows
FROM order_payments
""").df()


,total_rows,distinct_rows,duplicate_rows
0,103886,103886,0


In [33]:
dups_key = con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (order_id, payment_sequential)) AS distinct_key_rows,
  COUNT(*) - COUNT(DISTINCT (order_id, payment_sequential)) AS duplicate_key_rows
FROM order_payments
""").df()
dups_key


,total_rows,distinct_key_rows,duplicate_key_rows
0,103886,103886,0


In [34]:
out_path_order_pay = (PROCESSED_DIR / "order_payments_clean.csv").as_posix()

con.execute(f"""
COPY (
  SELECT *
  FROM order_payments
) TO '{out_path_order_pay}' (FORMAT CSV, HEADER, DELIMITER ',');
""")

Comme les resultats des analyses montrent aucune valeur manquante et aucun doublon alors on a juste à copier le fichier de base dans data/processed. C'est ce qu'on a fait.

- Passons à la table order_reviews

In [35]:
con = duckdb.connect()

csv_path_order_rev = RAW_DIR / "olist_order_reviews_dataset.csv"

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_order_rev.as_posix()}')
    LIMIT 5
""").df()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [36]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE order_reviews AS
SELECT *
FROM read_csv_auto('{csv_path_order_rev.as_posix()}');
""")
con.execute("""
SELECT
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS review_id_nulls,
  SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS review_score_nulls,
  SUM(CASE WHEN review_comment_title IS NULL THEN 1 ELSE 0 END) AS review_comment_title_nulls,
  SUM(CASE WHEN review_comment_message IS NULL THEN 1 ELSE 0 END) AS review_comment_message_nulls
FROM order_reviews
""").df()

,order_id_nulls,review_id_nulls,review_score_nulls,review_comment_title_nulls,review_comment_message_nulls
0,0.0,0.0,0.0,87656.0,58247.0


In [37]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,

  SUM(CASE WHEN review_comment_title IS NULL THEN 1 ELSE 0 END) AS review_comment_title_nulls,
  ROUND(
    100.0 * SUM(CASE WHEN review_comment_title IS NULL THEN 1 ELSE 0 END) / COUNT(*),
    2
  ) AS review_comment_title_null_pct,

  SUM(CASE WHEN review_comment_message IS NULL THEN 1 ELSE 0 END) AS review_comment_message_nulls,
  ROUND(
    100.0 * SUM(CASE WHEN review_comment_message IS NULL THEN 1 ELSE 0 END) / COUNT(*),
    2
  ) AS review_comment_message_null_pct

FROM order_reviews
""").df()


,total_rows,review_comment_title_nulls,review_comment_title_null_pct,review_comment_message_nulls,review_comment_message_null_pct
0,99224,87656.0,88.34,58247.0,58.7


On respectivement environ 88% et 58% de valeurs manquantes pour les colonnes review_comment_title et review_comment message, je pense qu'il serait preferable de supprimer ces deux variables. Mais vu qu'il est possible qu'on change de decision pour effectuer du NLP, on decide alors de le laisser.

In [38]:
con.execute("""
CREATE OR REPLACE TEMP TABLE order_reviews_clean AS
SELECT
  order_id,
  review_id,
  review_score,

 
  CASE
    WHEN review_comment_title IS NOT NULL
      OR review_comment_message IS NOT NULL
    THEN 1
    ELSE 0
  END AS has_text,

  TRIM(
    COALESCE(review_comment_title, '') || ' ' ||
    COALESCE(review_comment_message, '')
  ) AS review_text

FROM order_reviews
""")

con.execute(f"""
COPY ( 
    SELECT *
    FROM order_reviews
    ) TO '{(PROCESSED_DIR / "order_reviews_clean.csv").as_posix()}' (FORMAT CSV, HEADER, DELIMITER ',');
    """)

con.execute(f"""
    SELECT *
    FROM order_reviews_clean
    LIMIT 5
""").df()

,order_id,review_id,review_score,has_text,review_text
0,73fc7af87114b39712e6da79b0a377eb,7bc2406110b926393aa56f80a40eba40,4,0,
1,a548910a1c6147796b98fdf73dbeba33,80e641a11e56f04c1ad469d5645fdfde,5,0,
2,f9e4b658b201a9f2ecdecbb34bed034b,228ce5500dc1d8e020d8d1322874b6f0,5,0,
3,658677c97b385a9be170737859d3511b,e64fb393e7b32834bb789ff8bb30750e,5,1,Recebi bem antes do prazo estipulado.
4,8e6bfb81e283fa7e4f11123a3fb894f1,f7c4243c7fe1938f181bec41a392bdeb,5,1,Parabéns lojas lannister adorei comprar pela I...


Nous venons de creer une nouvelle colonne has_text qui donne 1 s'il y a un texte et 0 sinon. De plus, nous avons associé le titre + Commentaire pour former une variable review_text.

- Passons à la table order

In [39]:
con = duckdb.connect()

csv_path_order = RAW_DIR / "olist_orders_dataset.csv"

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_order.as_posix()}')
    LIMIT 5
""").df()

con.execute(f"""
CREATE OR REPLACE TEMP TABLE orders AS
SELECT *
FROM read_csv_auto('{csv_path_order.as_posix()}');
""")

In [40]:
df = con.execute("""
SELECT
  COUNT(*) AS total_rows,

  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS order_status_nulls,
  SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS order_approved_at_nulls,
  SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS order_delivered_carrier_date_nulls,
  SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS order_delivered_customer_date_nulls,

  ROUND(100.0 * SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS order_id_null_pct,
  ROUND(100.0 * SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS customer_id_null_pct,
  ROUND(100.0 * SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS order_status_null_pct,
  ROUND(100.0 * SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS order_approved_at_null_pct,
  ROUND(100.0 * SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS order_delivered_carrier_date_null_pct,
  ROUND(100.0 * SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS order_delivered_customer_date_null_pct

FROM orders
""").df()

df


,total_rows,order_id_nulls,customer_id_nulls,order_status_nulls,order_approved_at_nulls,order_delivered_carrier_date_nulls,order_delivered_customer_date_nulls,order_id_null_pct,customer_id_null_pct,order_status_null_pct,order_approved_at_null_pct,order_delivered_carrier_date_null_pct,order_delivered_customer_date_null_pct
0,99441,0.0,0.0,0.0,160.0,1783.0,2965.0,0.0,0.0,0.0,0.1609,1.793,2.9817


Nous remarquons quelques variables manquantes. mais ces lignes ne valent moins de 5% des valeurs manquantes. nous pouvons juger de les supprimer ou de les remplir par imputation en calculant les medianes de differents delais.

In [41]:
con.execute("""
WITH delays AS (
  SELECT
    median(order_approved_at - order_purchase_timestamp) AS med_approve_delay,
    median(order_delivered_carrier_date - order_approved_at) AS med_carrier_delay,
    median(order_delivered_customer_date - order_delivered_carrier_date) AS med_customer_delay
  FROM orders
  WHERE
    order_approved_at IS NOT NULL
    AND order_delivered_carrier_date IS NOT NULL
    AND order_delivered_customer_date IS NOT NULL
)
SELECT * FROM delays;
""").df()

,med_approve_delay,med_carrier_delay,med_customer_delay
0,0 days 00:20:36,1 days 19:34:50,7 days 02:23:34


In [42]:
con.execute ("""
    CREATE OR REPLACE TABLE orders_imputed AS
WITH delays AS (
  SELECT
    median(order_approved_at - order_purchase_timestamp) AS med_approve_delay,
    median(order_delivered_carrier_date - order_approved_at) AS med_carrier_delay,
    median(order_delivered_customer_date - order_delivered_carrier_date) AS med_customer_delay
  FROM orders
  WHERE
    order_approved_at IS NOT NULL
    AND order_delivered_carrier_date IS NOT NULL
    AND order_delivered_customer_date IS NOT NULL
)
SELECT
  *,
  COALESCE(
    order_approved_at,
    order_purchase_timestamp + delays.med_approve_delay
  ) AS order_approved_at_imp,

  COALESCE(
    order_delivered_carrier_date,
    COALESCE(order_approved_at, order_purchase_timestamp + delays.med_approve_delay)
    + delays.med_carrier_delay
  ) AS order_delivered_carrier_date_imp,

  COALESCE(
    order_delivered_customer_date,
    COALESCE(order_delivered_carrier_date,
      COALESCE(order_approved_at, order_purchase_timestamp + delays.med_approve_delay)
      + delays.med_carrier_delay
    )
    + delays.med_customer_delay
  ) AS order_delivered_customer_date_imp

FROM orders, delays;
""")

con.execute("""
select * from orders_imputed limit 5
""").df()



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,med_approve_delay,med_carrier_delay,med_customer_delay,order_approved_at_imp,order_delivered_carrier_date_imp,order_delivered_customer_date_imp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0 days 00:20:36,1 days 19:34:50,7 days 02:23:34,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0 days 00:20:36,1 days 19:34:50,7 days 02:23:34,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0 days 00:20:36,1 days 19:34:50,7 days 02:23:34,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0 days 00:20:36,1 days 19:34:50,7 days 02:23:34,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0 days 00:20:36,1 days 19:34:50,7 days 02:23:34,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02


In [43]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,

  SUM(order_approved_at_imp IS NULL) AS approved_imp_nulls,
  SUM(order_delivered_carrier_date_imp IS NULL) AS carrier_imp_nulls,
  SUM(order_delivered_customer_date_imp IS NULL) AS customer_imp_nulls
FROM orders_imputed;
""").df()


,total_rows,approved_imp_nulls,carrier_imp_nulls,customer_imp_nulls
0,99441,0.0,0.0,0.0


Bien joué, là on se rend compte qu'on n'a plus de valeurs manquantes. Ensuite verifions si cette table contient des doublons.

In [44]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS distinct_order_id,
  COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_id_rows
FROM orders_imputed;
""").df()


,total_rows,distinct_order_id,duplicate_order_id_rows
0,99441,99441,0


On remarque également aucun doublon sur cette table. Par consequent, nous pourrons l'enregistrer dans notre base.

In [45]:
con.execute(f"""
COPY ( 
    SELECT *
    FROM orders_imputed
    ) TO '{(PROCESSED_DIR / "orders_clean.csv").as_posix()}' (FORMAT CSV, HEADER, DELIMITER ',');
    """)

Passons à la table product 

In [46]:
con = duckdb.connect()

csv_path_product = RAW_DIR / "olist_products_dataset.csv"

con.execute(f"""
CREATE OR REPLACE TEMP TABLE products AS
SELECT *
FROM read_csv_auto('{csv_path_product.as_posix()}');
""")

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_product.as_posix()}')
    LIMIT 5
""").df()



,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [47]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  SUM(product_id IS NULL) AS product_id_nulls,
  SUM(product_category_name IS NULL) AS product_category_name_nulls,
  SUM(product_name_lenght IS NULL) AS product_name_lenght_nulls,
  SUM(product_description_lenght IS NULL) AS product_description_lenght_nulls,
  SUM(product_photos_qty IS NULL) AS product_photos_qty_nulls,
  SUM(product_weight_g IS NULL) AS product_weight_g_nulls
FROM products;
""").df()


,total_rows,product_id_nulls,product_category_name_nulls,product_name_lenght_nulls,product_description_lenght_nulls,product_photos_qty_nulls,product_weight_g_nulls
0,32951,0.0,610.0,610.0,610.0,610.0,2.0


In [48]:
con.execute("""
CREATE OR REPLACE TABLE products_clean AS
SELECT *
FROM products
WHERE NOT (
  product_category_name IS NULL
  AND product_name_lenght IS NULL
  AND product_description_lenght IS NULL
  AND product_photos_qty IS NULL
);
""")


In [49]:
con.execute("""
CREATE OR REPLACE TABLE products_clean AS
WITH med AS (
  SELECT median(product_weight_g) AS med_weight
  FROM products_clean
  WHERE product_weight_g IS NOT NULL
)
SELECT
  p.* EXCLUDE(product_weight_g),
  COALESCE(p.product_weight_g, med.med_weight) AS product_weight_g
FROM products_clean p, med;
""")


In [50]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  SUM(product_category_name IS NULL) AS product_category_name_nulls,
  SUM(product_name_lenght IS NULL) AS product_name_lenght_nulls,
  SUM(product_description_lenght IS NULL) AS product_description_lenght_nulls,
  SUM(product_photos_qty IS NULL) AS product_photos_qty_nulls,
  SUM(product_weight_g IS NULL) AS product_weight_g_nulls
FROM products_clean;
""").df()



,total_rows,product_category_name_nulls,product_name_lenght_nulls,product_description_lenght_nulls,product_photos_qty_nulls,product_weight_g_nulls
0,32341,0.0,0.0,0.0,0.0,0.0


In [51]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT product_id) AS distinct_product_id,
  COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_id_rows
FROM products;
""").df()


,total_rows,distinct_product_id,duplicate_product_id_rows
0,32951,32951,0


Là on voit bien evidemment que les colonnes ne contiennent plus des valeurs manquantes ni de doublons et par consequent nous l'enregistrons dans notre dossier data/processed.

In [52]:
con.execute(f"""
COPY ( 
    SELECT *
    FROM products_clean
    ) TO '{(PROCESSED_DIR / "products_clean.csv").as_posix()}' (FORMAT CSV, HEADER, DELIMITER ',');
    """)

- Et enfin, nous finissons le traitement de ces tables avec la table sellers.

In [53]:
con = duckdb.connect()

csv_path_sellers = RAW_DIR / "olist_sellers_dataset.csv"

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sellers AS
SELECT *
FROM read_csv_auto('{csv_path_sellers.as_posix()}');
""")

con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path_sellers.as_posix()}')
    LIMIT 5
""").df()


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [54]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  SUM(seller_id IS NULL) AS seller_id_nulls,
  SUM(seller_city IS NULL) AS seller_city_nulls,
  SUM(seller_state IS NULL) AS seller_state_nulls,
  SUM(seller_zip_code_prefix IS NULL) AS seller_zip_code_prefix_nulls
FROM sellers;
""").df()


,total_rows,seller_id_nulls,seller_city_nulls,seller_state_nulls,seller_zip_code_prefix_nulls
0,3095,0.0,0.0,0.0,0.0


In [55]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT seller_id) AS distinct_seller_id,
  COUNT(*) - COUNT(DISTINCT seller_id) AS duplicate_seller_id_rows
FROM sellers;
""").df()

,total_rows,distinct_seller_id,duplicate_seller_id_rows
0,3095,3095,0


Pareil pour cette dernière table, pas de valeurs manquantes ni de doublons donc nous passerons à l'enregistrement.

In [56]:
con.execute(f"""
COPY ( 
    SELECT *
    FROM sellers
    ) TO '{(PROCESSED_DIR / "sellers_clean.csv").as_posix()}' (FORMAT CSV, HEADER, DELIMITER ',');
    """)

Pour l'autre fichier restant, il s'agit uniquement de la traduction de noms de produits en anglais dont nous ne nous attarderons pas la dessus.